# Copilot Phase 0 — Monthly Operational Snapshot

**Goal:** Simulate the report a bank risk team pulls at month-end — 
*"As of this month, how healthy is our loan portfolio?"*

This notebook ingests the raw `payment_history`, `loans`, and `borrowers` tables 
and produces two output datasets:

| Output file | Granularity | Description |
|---|---|---|
| `loan_monthly_snapshot.csv` | one row = one loan × one month | loan-level cumulative KPIs |
| `portfolio_monthly_kpi.csv` | one row = one month | portfolio-level KPI summary |

> **Scope (Phase 0 / Step 1):** loan-level cumulative snapshot — `on_time_rate_cumul` and `total_delinquent_cumul`.
> Steps 2 & 3 (portfolio KPIs + CSV export) follow in subsequent cells.

## 0 · Imports & Config

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_DIR = Path('.')   # notebook lives next to the CSVs
print('pandas', pd.__version__, '| numpy', np.__version__)

pandas 2.3.3 | numpy 2.3.5


## 1 · Load Raw Tables

In [2]:
payment_history = pd.read_csv(DATA_DIR / 'payment_history.csv')
loans           = pd.read_csv(DATA_DIR / 'loans.csv')
borrowers       = pd.read_csv(DATA_DIR / 'borrowers.csv')

print(f'payment_history : {payment_history.shape[0]:,} rows × {payment_history.shape[1]} cols')
print(f'loans           : {loans.shape[0]:,} rows × {loans.shape[1]} cols')
print(f'borrowers       : {borrowers.shape[0]:,} rows × {borrowers.shape[1]} cols')
payment_history.head()

payment_history : 627,511 rows × 6 cols
loans           : 181,800 rows × 9 cols
borrowers       : 71,681 rows × 10 cols


,loan_id,month_number,payment_due_date,amount_due,amount_paid,days_delinquent
0,L77639-244888,1,2023-01-15,2568.3900,2073.5000,0.0000
1,L77639-244888,2,2023-02-15,2568.3900,2586.1600,0.0000
2,L77639-244888,3,2023-03-15,2568.3900,2138.3500,0.0000
3,L76536-178626,1,2023-01-15,2468.5700,2219.5500,0.0000
4,L76536-178626,2,2023-02-15,2468.5700,2217.1500,0.0000


### 1.1 Schema & null audit

In [3]:
print('=== payment_history dtypes & nulls ===')
print(payment_history.dtypes)
print()
print(payment_history.isnull().sum())
print()
print('month_number range:', payment_history.month_number.min(), '–', payment_history.month_number.max())
print('unique loans      :', payment_history.loan_id.nunique())
print('rows per loan     :')
print(payment_history.groupby('loan_id')['month_number'].count().value_counts().sort_index())

=== payment_history dtypes & nulls ===
loan_id              object
month_number          int64
payment_due_date     object
amount_due          float64
amount_paid         float64
days_delinquent     float64
dtype: object

loan_id                 0
month_number            0
payment_due_date    15537
amount_due          15480
amount_paid         15578
days_delinquent     15390
dtype: int64

month_number range: 1 – 7
unique loans      : 180000
rows per loan     :
month_number
1      261
2    12152
3    78980
4    77253
5    11132
6      220
7        2
Name: count, dtype: int64


---
## Step 1 · Loan-Level Cumulative Snapshot

For every `(loan_id, month_number)` pair we compute **expanding-window** aggregates 
— i.e. metrics that cover **month 1 through the current month** inclusive.

| Column | Definition |
|---|---|
| `on_time_flag` | 1 if `days_delinquent == 0` (or null, treated as 0), else 0 |
| `on_time_rate_cumul` | cumulative on-time payments / cumulative total payments |
| `total_delinquent_cumul` | cumulative sum of `days_delinquent` up to this month |
| `months_observed` | how many months of history exist up to this point |
| `months_delinquent_cumul` | count of months where `days_delinquent > 0` |
| `worst_delinquent_month` | worst single-month `days_delinquent` seen so far |

### 1.1 Derive `on_time_flag` from `days_delinquent`

In [4]:
# Sort so the expanding window processes months in order
ph = payment_history.sort_values(['loan_id', 'month_number']).copy()

# Treat null days_delinquent as 0 (payment record exists but delinquency not recorded)
ph['days_delinquent_filled'] = ph['days_delinquent'].fillna(0.0)

# A payment is on-time when there are zero delinquent days
ph['on_time_flag'] = (ph['days_delinquent_filled'] == 0).astype(int)

print('on_time_flag distribution:')
print(ph['on_time_flag'].value_counts())
print(f'\nOverall on-time rate: {ph["on_time_flag"].mean():.4f}')

on_time_flag distribution:
on_time_flag
1    589282
0     38229
Name: count, dtype: int64

Overall on-time rate: 0.9391


### 1.2 Expanding-window cumulative metrics

We use `groupby + expanding()` so that for loan L001 at month 3 we only look at months 1–3, 
not months 4–7.  This faithfully mimics a real-time monitoring system.

In [5]:
grp = ph.groupby('loan_id', sort=False)

# -- cumulative counts & sums (expanding window) --
ph['on_time_count_cumul']      = grp['on_time_flag'].cumsum()          # payments on time so far
ph['months_observed']          = grp['on_time_flag'].cumcount() + 1    # row index within group (1-based)
ph['total_delinquent_cumul']   = grp['days_delinquent_filled'].cumsum() # cumulative delinquent days
ph['months_delinquent_cumul']  = grp['on_time_flag'].transform(
    lambda s: (1 - s).cumsum()
)                                                                        # months with any delinquency
ph['worst_delinquent_month']   = grp['days_delinquent_filled'].transform(
    lambda s: s.expanding().max()
)                                                                        # worst single month so far

# -- on_time_rate_cumul: on-time months / total months observed --
ph['on_time_rate_cumul'] = ph['on_time_count_cumul'] / ph['months_observed']

print('Sample expanding metrics for loan', ph['loan_id'].iloc[0])
cols_show = ['loan_id','month_number','days_delinquent_filled','on_time_flag',
             'months_observed','on_time_count_cumul','on_time_rate_cumul',
             'total_delinquent_cumul','months_delinquent_cumul','worst_delinquent_month']
ph[cols_show].head(10)

Sample expanding metrics for loan L100-29525


,loan_id,month_number,days_delinquent_filled,on_time_flag,months_observed,on_time_count_cumul,on_time_rate_cumul,total_delinquent_cumul,months_delinquent_cumul,worst_delinquent_month
373361,L100-29525,1,0.0000,1,1,1,1.0000,0.0000,0,0.0000
373362,L100-29525,2,0.0000,1,2,2,1.0000,0.0000,0,0.0000
373363,L100-29525,3,0.0000,1,3,3,1.0000,0.0000,0,0.0000
527910,L100-526649,1,0.0000,1,1,1,1.0000,0.0000,0,0.0000
527911,L100-526649,2,0.0000,1,2,2,1.0000,0.0000,0,0.0000
527912,L100-526649,3,0.0000,1,3,3,1.0000,0.0000,0,0.0000
527913,L100-526649,4,0.0000,1,4,4,1.0000,0.0000,0,0.0000
527914,L100-526649,5,0.0000,1,5,5,1.0000,0.0000,0,0.0000
154644,L100-874457,1,0.0000,1,1,1,1.0000,0.0000,0,0.0000
154645,L100-874457,2,0.0000,1,2,2,1.0000,0.0000,0,0.0000


### 1.3 Month-over-month change in `on_time_rate_cumul`

Captures whether a loan is **deteriorating** — its cumulative on-time rate is falling. 
This feeds the *new deteriorating loans* KPI in Step 2.

In [6]:
ph['on_time_rate_mom_change'] = (
    ph.groupby('loan_id')['on_time_rate_cumul']
      .diff()          # current month − previous month (NaN for month 1)
)

# Flag: rate dropped by more than 10 percentage points vs last month
DETERIORATION_THRESHOLD = -0.10
ph['is_deteriorating'] = (
    (ph['on_time_rate_mom_change'] < DETERIORATION_THRESHOLD)
).astype(int)

print('is_deteriorating distribution:', ph['is_deteriorating'].value_counts().to_dict())
print(f'Deteriorating observations: {ph["is_deteriorating"].sum():,} '
      f'({ph["is_deteriorating"].mean()*100:.2f}% of all loan-months)')

is_deteriorating distribution: {0: 594294, 1: 33217}
Deteriorating observations: 33,217 (5.29% of all loan-months)


### 1.4 Attach loan metadata

Join `loan_type` and `state` (via borrowers) so downstream groupings work.

In [7]:
loan_meta = (
    loans[['loan_id','loan_type','borrower_id','principal','interest_rate',
            'term_months','monthly_payment','origination_date']]
    .merge(
        borrowers[['borrower_id','state','credit_score','annual_income']],
        on='borrower_id', how='left'
    )
)

snap = ph.merge(loan_meta, on='loan_id', how='left')

print('loan_monthly_snapshot shape:', snap.shape)
print('null counts in key cols:')
key_cols = ['loan_type','state','credit_score','annual_income']
print(snap[key_cols].isnull().sum())

loan_monthly_snapshot shape: (640096, 26)
null counts in key cols:
loan_type        5403
state            8218
credit_score     7977
annual_income    8766
dtype: int64


### 1.5 Select & rename final snapshot columns

Keep only the columns needed for the dashboard and AI insight engine.

In [8]:
loan_monthly_snapshot = snap[[
    # identifiers
    'loan_id', 'borrower_id', 'month_number',
    # loan attributes
    'loan_type', 'state', 'principal', 'interest_rate',
    'term_months', 'monthly_payment', 'origination_date',
    'credit_score', 'annual_income',
    # current-month payment details
    'amount_due', 'amount_paid', 'days_delinquent_filled',
    'on_time_flag',
    # cumulative KPIs (expanding window)
    'months_observed',
    'on_time_rate_cumul',
    'total_delinquent_cumul',
    'months_delinquent_cumul',
    'worst_delinquent_month',
    # trend
    'on_time_rate_mom_change',
    'is_deteriorating',
]].rename(columns={'days_delinquent_filled': 'days_delinquent'})

print('Final snapshot shape:', loan_monthly_snapshot.shape)
loan_monthly_snapshot.head(10)

Final snapshot shape: (640096, 23)


,loan_id,borrower_id,month_number,loan_type,state,principal,interest_rate,term_months,monthly_payment,origination_date,credit_score,annual_income,amount_due,amount_paid,days_delinquent,on_time_flag,months_observed,on_time_rate_cumul,total_delinquent_cumul,months_delinquent_cumul,worst_delinquent_month,on_time_rate_mom_change,is_deteriorating
0,L100-29525,BR86818-341703,1,credit_line,CA,83620.4100,15.1500,42.0000,2577.5000,2024-02-27,576.0000,129497.6400,2577.5000,2350.9000,0.0000,1,1,1.0000,0.0000,0,0.0000,NaN,0
1,L100-29525,BR86818-341703,2,credit_line,CA,83620.4100,15.1500,42.0000,2577.5000,2024-02-27,576.0000,129497.6400,2577.5000,2181.5200,0.0000,1,2,1.0000,0.0000,0,0.0000,0.0000,0
2,L100-29525,BR86818-341703,3,credit_line,CA,83620.4100,15.1500,42.0000,2577.5000,2024-02-27,576.0000,129497.6400,2577.5000,1920.4900,0.0000,1,3,1.0000,0.0000,0,0.0000,0.0000,0
3,L100-526649,ORPHAN-95217-41138,1,home_equity,NaN,75007.7000,12.6200,61.0000,1672.1700,2024-07-24,NaN,NaN,1672.1700,1440.4900,0.0000,1,1,1.0000,0.0000,0,0.0000,NaN,0
4,L100-526649,BR6948-393872,1,home_equity,IN,75007.7000,12.6200,61.0000,1672.1700,2024-07-24,575.0000,176331.0300,1672.1700,1440.4900,0.0000,1,1,1.0000,0.0000,0,0.0000,NaN,0
5,L100-526649,ORPHAN-95217-41138,2,home_equity,NaN,75007.7000,12.6200,61.0000,1672.1700,2024-07-24,NaN,NaN,1672.1700,1425.3100,0.0000,1,2,1.0000,0.0000,0,0.0000,0.0000,0
6,L100-526649,BR6948-393872,2,home_equity,IN,75007.7000,12.6200,61.0000,1672.1700,2024-07-24,575.0000,176331.0300,1672.1700,1425.3100,0.0000,1,2,1.0000,0.0000,0,0.0000,0.0000,0
7,L100-526649,ORPHAN-95217-41138,3,home_equity,NaN,75007.7000,12.6200,61.0000,1672.1700,2024-07-24,NaN,NaN,1672.1700,1410.6600,0.0000,1,3,1.0000,0.0000,0,0.0000,0.0000,0
8,L100-526649,BR6948-393872,3,home_equity,IN,75007.7000,12.6200,61.0000,1672.1700,2024-07-24,575.0000,176331.0300,1672.1700,1410.6600,0.0000,1,3,1.0000,0.0000,0,0.0000,0.0000,0
9,L100-526649,ORPHAN-95217-41138,4,home_equity,NaN,75007.7000,12.6200,61.0000,1672.1700,2024-07-24,NaN,NaN,1672.1700,1522.7500,0.0000,1,4,1.0000,0.0000,0,0.0000,0.0000,0


### 1.6 Sanity checks

Validate the expanding-window logic on a few example loans.

In [9]:
# Pick a loan that has at least one delinquent month to make the test interesting
delinquent_loans = (
    loan_monthly_snapshot
    .groupby('loan_id')['days_delinquent']
    .max()
    .loc[lambda s: s > 0]
    .index
)
example_loan = delinquent_loans[0]
print(f'Example loan with delinquency: {example_loan}')

check_cols = ['month_number','days_delinquent','on_time_flag',
              'months_observed','on_time_rate_cumul',
              'total_delinquent_cumul','on_time_rate_mom_change','is_deteriorating']
print(loan_monthly_snapshot
      .loc[loan_monthly_snapshot['loan_id'] == example_loan, check_cols]
      .to_string(index=False))

# Assert: on_time_rate_cumul is always in [0, 1]
assert loan_monthly_snapshot['on_time_rate_cumul'].between(0, 1).all(), \
    'on_time_rate_cumul out of [0,1]'

# Assert: total_delinquent_cumul is non-negative and non-decreasing within each loan
monotone = (
    loan_monthly_snapshot
    .sort_values(['loan_id','month_number'])
    .groupby('loan_id')['total_delinquent_cumul']
    .apply(lambda s: (s.diff().fillna(0) >= 0).all())
    .all()
)
assert monotone, 'total_delinquent_cumul is not monotonically non-decreasing'

print('\n✓ All sanity checks passed.')

Example loan with delinquency: L10000-291243
 month_number  days_delinquent  on_time_flag  months_observed  on_time_rate_cumul  total_delinquent_cumul  on_time_rate_mom_change  is_deteriorating
            1           0.0000             1                1              1.0000                  0.0000                      NaN                 0
            2           0.0000             1                2              1.0000                  0.0000                   0.0000                 0
            3           5.0000             0                3              0.6667                  5.0000                  -0.3333                 1
            4           1.0000             0                4              0.5000                  6.0000                  -0.1667                 1

✓ All sanity checks passed.


### 1.7 Distribution preview — cumulative KPIs at final observed month

For each loan, look at the KPIs as of its **last recorded month** — 
this represents the most current risk reading per loan.

In [10]:
latest = (
    loan_monthly_snapshot
    .sort_values(['loan_id','month_number'])
    .groupby('loan_id')
    .last()
    .reset_index()
)

print('=== on_time_rate_cumul @ final month ===')
print(latest['on_time_rate_cumul'].describe())
print()
print('=== total_delinquent_cumul @ final month ===')
print(latest['total_delinquent_cumul'].describe())
print()
# Bucket on_time_rate into risk bands
bins   = [0, 0.5, 0.7, 0.8, 0.9, 1.0001]
labels = ['Critical (<50%)', 'High (50-70%)', 'Medium (70-80%)',
          'Low (80-90%)', 'Healthy (90-100%)']
latest['risk_band'] = pd.cut(latest['on_time_rate_cumul'], bins=bins, labels=labels, right=False)
print('=== Risk band distribution (as of last observed month) ===')
print(latest['risk_band'].value_counts().sort_index())

=== on_time_rate_cumul @ final month ===
count   180000.0000
mean         0.9276
std          0.1899
min          0.0000
25%          1.0000
50%          1.0000
75%          1.0000
max          1.0000
Name: on_time_rate_cumul, dtype: float64

=== total_delinquent_cumul @ final month ===
count   180000.0000
mean         0.5402
std          1.6471
min          0.0000
25%          0.0000
50%          0.0000
75%          0.0000
max         24.0000
Name: total_delinquent_cumul, dtype: float64

=== Risk band distribution (as of last observed month) ===
risk_band
Critical (<50%)        8458
High (50-70%)         14649
Medium (70-80%)        3744
Low (80-90%)            157
Healthy (90-100%)    152992
Name: count, dtype: int64


---
## ✓ Step 1 Complete

The `loan_monthly_snapshot` DataFrame is ready:  
**{rows:,} rows × {cols} cols** — one row per loan per month, 
with expanding-window cumulative KPIs.

**Next:** Step 2 — aggregate to portfolio-level monthly KPIs → `portfolio_monthly_kpi`.

In [11]:
print(f'loan_monthly_snapshot: {loan_monthly_snapshot.shape[0]:,} rows × {loan_monthly_snapshot.shape[1]} cols')
print('Columns:')
for c in loan_monthly_snapshot.columns:
    print(' ', c)

loan_monthly_snapshot: 640,096 rows × 23 cols
Columns:
  loan_id
  borrower_id
  month_number
  loan_type
  state
  principal
  interest_rate
  term_months
  monthly_payment
  origination_date
  credit_score
  annual_income
  amount_due
  amount_paid
  days_delinquent
  on_time_flag
  months_observed
  on_time_rate_cumul
  total_delinquent_cumul
  months_delinquent_cumul
  worst_delinquent_month
  on_time_rate_mom_change
  is_deteriorating


---
## Step 2 · Portfolio-Level Monthly KPI Aggregation

Collapse `loan_monthly_snapshot` into the numbers a risk management team 
reviews in their monthly meeting. Aggregation key: `month_number` (1–7), 
which maps cleanly to calendar months for the bulk of the portfolio.

| Output | Rows | Description |
|---|---|---|
| `portfolio_monthly_kpi.csv` | 7 | overall portfolio, one row per month |
| `portfolio_monthly_kpi_by_segment.csv` | ~1 000 | by `loan_type` and by `state` |

| KPI | Definition |
|---|---|
| `active_loans` | loans with a payment record in that month |
| `total_principal_outstanding` | sum of `principal` for active loans |
| `avg_on_time_rate` | mean `on_time_rate_cumul` across active loans |
| `median_on_time_rate` | median — more robust to outliers |
| `pct_delinquent` | share of loans with `on_time_rate_cumul < 1.0` |
| `pct_high_risk` | share of loans with `on_time_rate_cumul < 0.6` |
| `deteriorating_count` | loans where `is_deteriorating = 1` this month |
| `deteriorating_pct` | `deteriorating_count / active_loans` |

### 2.1 Derive `calendar_month` and normalize segment dimensions

- **calendar_month**: parsed from `payment_due_date`; falls back to `origination_date + (month_number − 1) × 30d` for the ~2.5 % of null dates.
- **loan_type**: raw data has case variants (`auto`, `Auto`, `AUTO`) — normalize to lowercase-stripped.
- **state / loan_type nulls**: filled as `'unknown'` so segment KPIs still sum to portfolio totals.

In [ ]:
snap = loan_monthly_snapshot.copy()

# -- calendar_month --------------------------------------------------------
ph_dates = payment_history[['loan_id','month_number','payment_due_date']].copy()
ph_dates['payment_due_date'] = pd.to_datetime(ph_dates['payment_due_date'], errors='coerce')
snap = snap.merge(ph_dates, on=['loan_id','month_number'], how='left')

snap['origination_date'] = pd.to_datetime(snap['origination_date'], errors='coerce')
fallback_date = snap['origination_date'] + pd.to_timedelta(
    (snap['month_number'] - 1) * 30, unit='D'
)
snap['payment_due_date'] = snap['payment_due_date'].fillna(fallback_date)
snap['calendar_month']   = snap['payment_due_date'].dt.to_period('M').astype(str)

# Drop the rare rows where even the fallback produces NaT (< 0.02 %)
snap = snap.dropna(subset=['calendar_month'])

# -- Normalize segment dimensions ------------------------------------------
snap['loan_type'] = snap['loan_type'].str.strip().str.lower().fillna('unknown')
snap['state']     = snap['state'].str.strip().fillna('Unknown')

print('calendar_month distribution (representative sample):')
print(snap['calendar_month'].value_counts().sort_index().head(10))
print(f'\nnull calendar_month after fill: {snap["calendar_month"].isnull().sum()}')
print('\nloan_type values after normalization:', sorted(snap['loan_type'].unique()))

### 2.2 Add binary helper columns for vectorized aggregation

Using `groupby.agg()` with pre-computed binary columns is **~50× faster** than 
`groupby.apply(custom_fn)` on 640 K rows — same result, no timeout risk.

In [ ]:
snap['is_delinquent'] = (snap['on_time_rate_cumul'] < 1.0).astype(float)
snap['is_high_risk']  = (snap['on_time_rate_cumul'] < 0.6).astype(float)

print('is_delinquent rate:', round(snap['is_delinquent'].mean(), 4))
print('is_high_risk rate :', round(snap['is_high_risk'].mean(), 4))

### 2.3 KPI aggregation helper

In [ ]:
def agg_kpis(df: 'pd.DataFrame', key) -> 'pd.DataFrame':
    """Vectorized groupby.agg over an arbitrary grouping key."""
    result = df.groupby(key, sort=True).agg(
        active_loans                = ('loan_id',            'count'),
        total_principal_outstanding = ('principal',          'sum'),
        avg_on_time_rate            = ('on_time_rate_cumul', 'mean'),
        median_on_time_rate         = ('on_time_rate_cumul', 'median'),
        pct_delinquent              = ('is_delinquent',      'mean'),
        pct_high_risk               = ('is_high_risk',       'mean'),
        deteriorating_count         = ('is_deteriorating',   'sum'),
        deteriorating_pct           = ('is_deteriorating',   'mean'),
    ).reset_index()
    float_cols = ['avg_on_time_rate','median_on_time_rate',
                  'pct_delinquent','pct_high_risk','deteriorating_pct']
    result[float_cols] = result[float_cols].round(4)
    result['total_principal_outstanding'] = (
        result['total_principal_outstanding'].round(0).astype(int)
    )
    result['deteriorating_count'] = result['deteriorating_count'].astype(int)
    return result

print('agg_kpis() defined.')

### 2.4 Overall portfolio KPI — one row per month

In [ ]:
portfolio_monthly_kpi = agg_kpis(snap, 'month_number')

# Attach representative calendar month (majority-vote from actual payment dates)
cal_map = (
    snap.groupby('month_number')['calendar_month']
    .agg(lambda x: x.value_counts().index[0])
    .rename('representative_month')
    .reset_index()
)
portfolio_monthly_kpi = portfolio_monthly_kpi.merge(cal_map, on='month_number')

# Reorder: identifiers first
lead = ['month_number', 'representative_month']
portfolio_monthly_kpi = portfolio_monthly_kpi[
    lead + [c for c in portfolio_monthly_kpi.columns if c not in lead]
]

print(f'portfolio_monthly_kpi: {portfolio_monthly_kpi.shape}')
display_cols = ['month_number','representative_month','active_loans',
                'avg_on_time_rate','pct_delinquent','pct_high_risk',
                'deteriorating_count','deteriorating_pct']
portfolio_monthly_kpi[display_cols]

### 2.5 Segment KPI — by `loan_type` × month

In [ ]:
kpi_by_loan_type = agg_kpis(snap, ['month_number','loan_type'])
kpi_by_loan_type['segment_type'] = 'loan_type'
kpi_by_loan_type['is_top10']     = float('nan')   # N/A for loan_type dimension
kpi_by_loan_type = kpi_by_loan_type.rename(columns={'loan_type': 'segment_value'})

print(f'kpi_by_loan_type: {kpi_by_loan_type.shape}')
print('Unique loan_types:', sorted(kpi_by_loan_type['segment_value'].unique()))

# Preview: month 3 breakdown
kpi_by_loan_type[
    kpi_by_loan_type['month_number'] == 3
][['segment_value','active_loans','avg_on_time_rate',
   'pct_high_risk','deteriorating_pct']].sort_values('active_loans', ascending=False)

### 2.6 Segment KPI — by `state` × month

All states are kept in the output (useful for choropleth maps). 
`is_top10` flags the states with the highest loan volume for quick-filter use.

In [ ]:
top10_states = (
    snap[snap['state'] != 'Unknown']
    .groupby('state')['loan_id']
    .count()
    .nlargest(10)
    .index.tolist()
)
print('Top-10 states by total loan volume:', top10_states)

kpi_by_state = agg_kpis(snap, ['month_number','state'])
kpi_by_state['segment_type'] = 'state'
kpi_by_state['is_top10']     = kpi_by_state['state'].isin(top10_states).astype(int)
kpi_by_state = kpi_by_state.rename(columns={'state': 'segment_value'})

print(f'kpi_by_state: {kpi_by_state.shape}')

# Preview: top-10 states at month 3
kpi_by_state[
    (kpi_by_state['month_number'] == 3) & (kpi_by_state['is_top10'] == 1)
][['segment_value','active_loans','avg_on_time_rate',
   'pct_high_risk','deteriorating_pct']].sort_values('active_loans', ascending=False)

### 2.7 Combine → `portfolio_monthly_kpi_by_segment`

In [ ]:
kpi_cols = [
    'active_loans','total_principal_outstanding','avg_on_time_rate',
    'median_on_time_rate','pct_delinquent','pct_high_risk',
    'deteriorating_count','deteriorating_pct',
]
seg_cols = ['month_number','segment_type','segment_value','is_top10'] + kpi_cols

portfolio_monthly_kpi_by_segment = pd.concat(
    [kpi_by_loan_type, kpi_by_state], ignore_index=True
)[seg_cols]

print(f'portfolio_monthly_kpi_by_segment: {portfolio_monthly_kpi_by_segment.shape}')
print('segment_type counts:')
print(portfolio_monthly_kpi_by_segment['segment_type'].value_counts())

### 2.8 Sanity checks

In [ ]:
# 1. Total active_loans across months == total rows in loan_monthly_snapshot
assert portfolio_monthly_kpi['active_loans'].sum() == len(snap), \
    'active_loans sum mismatch'

# 2. Rates in [0, 1]
assert portfolio_monthly_kpi['avg_on_time_rate'].between(0, 1).all()
assert portfolio_monthly_kpi['deteriorating_pct'].between(0, 1).all()

# 3. pct_high_risk ≤ pct_delinquent (high-risk ⊆ delinquent)
assert (portfolio_monthly_kpi['pct_high_risk']
        <= portfolio_monthly_kpi['pct_delinquent']).all()

# 4. loan_type segment active_loans cross-sums to portfolio total per month
lt_sum = (
    portfolio_monthly_kpi_by_segment
    .loc[portfolio_monthly_kpi_by_segment['segment_type'] == 'loan_type']
    .groupby('month_number')['active_loans'].sum()
)
port_totals = portfolio_monthly_kpi.set_index('month_number')['active_loans']
assert (lt_sum == port_totals).all(), 'loan_type active_loans cross-check failed'

print('✓ All 4 sanity checks passed.')

### 2.9 Export CSVs

In [ ]:
portfolio_monthly_kpi.to_csv(
    DATA_DIR / 'portfolio_monthly_kpi.csv', index=False
)
portfolio_monthly_kpi_by_segment.to_csv(
    DATA_DIR / 'portfolio_monthly_kpi_by_segment.csv', index=False
)

print('Saved:')
print(f'  portfolio_monthly_kpi.csv            — {len(portfolio_monthly_kpi)} rows')
print(f'  portfolio_monthly_kpi_by_segment.csv — {len(portfolio_monthly_kpi_by_segment)} rows')
print(f'    ├─ loan_type: '
      f'{(portfolio_monthly_kpi_by_segment.segment_type=="loan_type").sum()} rows')
print(f'    └─ state:     '
      f'{(portfolio_monthly_kpi_by_segment.segment_type=="state").sum()} rows')

---
## ✓ Step 2 Complete

| File | Rows | Ready for |
|---|---|---|
| `portfolio_monthly_kpi.csv` | 7 | overall trend lines, exec-level KPI card |
| `portfolio_monthly_kpi_by_segment.csv` | ~1 000 | loan_type bar charts, state choropleth |
| `loan_monthly_snapshot.csv` *(Step 1)* | 640 K | loan-level drill-down, sparklines |

**Next: Phase 1 — Dashboard** — these three files are the data layer.

In [ ]:
print('Phase 0 complete. Output files:')
for fname in ['loan_monthly_snapshot.csv',
              'portfolio_monthly_kpi.csv',
              'portfolio_monthly_kpi_by_segment.csv']:
    p = DATA_DIR / fname
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f'  ✓ {fname:<42} {size_kb:,.0f} KB')
    else:
        print(f'  ✗ {fname} NOT FOUND')